# RAG Evaluation

Notebook này có 3 cell để validate RAG nhanh và trực quan.

Workflow:

1. Nếu muốn sinh lại kết quả benchmark, chạy trước:

```bash
cd backend
python tests_local/test_rag_benchmark.py
```

2. Mở notebook này và `Run All`.

Luồng đọc notebook:

- **Cell 2 - Input check:** kiểm tra golden dataset, phân bố group/source/behavior.
- **Cell 3 - Output evaluation:** kiểm tra RAG output, metric tổng quan, metric theo nhóm, source recall, và case cần debug.

Notebook đọc `docs/rag_benchmark_dataset.json` và `docs/rag_benchmark_results.json`. Notebook không gọi trực tiếp API/LLM/Qdrant.

In [ ]:
# Cell 2 - Input check: golden dataset coverage
from pathlib import Path
import json
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "rag_benchmark_dataset.json").exists():
            return candidate
    raise FileNotFoundError("Cannot find docs/rag_benchmark_dataset.json from current directory")


def plot_bar(series: pd.Series, title: str, ylabel: str = "Count", color: str = "#2563eb", rotation: int = 0):
    ax = series.plot(kind="bar", color=color)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotation)
    for container in ax.containers:
        ax.bar_label(container, padding=3)
    plt.tight_layout()
    plt.show()


ROOT = find_repo_root()
DATASET_PATH = ROOT / "docs" / "rag_benchmark_dataset.json"
RESULT_PATH = ROOT / "docs" / "rag_benchmark_results.json"

with DATASET_PATH.open(encoding="utf-8") as f:
    dataset_df = pd.DataFrame(json.load(f))

required_dataset_columns = {
    "id", "group", "question", "ground_truth", "expected_source", "expected_behavior",
}
missing_dataset_columns = required_dataset_columns - set(dataset_df.columns)
duplicate_ids = dataset_df[dataset_df.duplicated("id", keep=False)].sort_values("id")

if missing_dataset_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_dataset_columns)}")
if not duplicate_ids.empty:
    display(duplicate_ids[["id", "group", "question"]])
    raise ValueError("Dataset contains duplicate ids")

display(Markdown(f"## Input check\nDataset: `{DATASET_PATH.relative_to(ROOT)}`  \nTotal questions: **{len(dataset_df)}**"))

dataset_summary = (
    dataset_df.groupby(["group", "expected_behavior"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["group", "expected_behavior"])
)
display(dataset_summary)

plot_bar(dataset_df["group"].value_counts().sort_index(), "Benchmark questions by group", color="#2563eb")
plot_bar(dataset_df["expected_source"].value_counts().sort_index(), "Expected source coverage", color="#059669")
plot_bar(dataset_df["expected_behavior"].value_counts().sort_values(ascending=False), "Expected behavior coverage", color="#7c3aed", rotation=45)

In [ ]:
# Cell 3 - Output evaluation: RAG answer quality and debug cases
def safe_mean(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna()
    return float(values.mean()) if not values.empty else 0.0


def normalize_sources(value) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if pd.isna(value):
        return []
    return [str(value)]


def plot_percent_bar(df: pd.DataFrame, x: str, y: str, title: str, color: str = "#2563eb", rotation: int = 0):
    ax = df.plot(kind="bar", x=x, y=y, legend=False, color=color)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Percent")
    ax.set_ylim(0, 105)
    ax.tick_params(axis="x", rotation=rotation)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", padding=3)
    plt.tight_layout()
    plt.show()


LOW_SCORE_THRESHOLD = 0.70
CASE_ID = "FAQ-04"
EXPORT_FAILURES = False
FAILURE_EXPORT_PATH = ROOT / "docs" / "rag_benchmark_failures.csv"

if not RESULT_PATH.exists():
    display(Markdown("## No benchmark results found\nRun `cd backend && python tests_local/test_rag_benchmark.py` first."))
else:
    with RESULT_PATH.open(encoding="utf-8") as f:
        results_df = pd.DataFrame(json.load(f))

    required_result_columns = {
        "id", "group", "question", "ground_truth", "predicted", "sources_returned",
        "expected_source", "expected_behavior", "http_status", "faithfulness",
        "relevance", "source_ok", "guardrail_pass",
    }
    missing_result_columns = required_result_columns - set(results_df.columns)
    if missing_result_columns:
        raise ValueError(f"Results are missing columns: {sorted(missing_result_columns)}")

    missing_result_ids = sorted(set(dataset_df["id"]) - set(results_df["id"]))
    extra_result_ids = sorted(set(results_df["id"]) - set(dataset_df["id"]))

    avg_faithfulness = safe_mean(results_df["faithfulness"])
    avg_relevance = safe_mean(results_df["relevance"])
    source_recall = safe_mean(results_df["source_ok"])
    guardrail_rows = results_df[(results_df["group"] == "guardrail") & results_df["guardrail_pass"].notna()]
    guardrail_rate = safe_mean(guardrail_rows["guardrail_pass"]) if not guardrail_rows.empty else 0.0
    overall_score = 0.35 * avg_faithfulness + 0.25 * avg_relevance + 0.20 * source_recall + 0.20 * guardrail_rate

    scorecard = pd.DataFrame([
        {"metric": "faithfulness", "score": avg_faithfulness, "score_pct": round(avg_faithfulness * 100, 2)},
        {"metric": "relevance", "score": avg_relevance, "score_pct": round(avg_relevance * 100, 2)},
        {"metric": "source_recall", "score": source_recall, "score_pct": round(source_recall * 100, 2)},
        {"metric": "guardrail_rate", "score": guardrail_rate, "score_pct": round(guardrail_rate * 100, 2)},
        {"metric": "overall_score", "score": overall_score, "score_pct": round(overall_score * 100, 2)},
    ])

    group_summary = (
        results_df.groupby("group")
        .agg(
            cases=("id", "count"),
            avg_faithfulness=("faithfulness", safe_mean),
            avg_relevance=("relevance", safe_mean),
            source_recall=("source_ok", safe_mean),
            guardrail_rate=("guardrail_pass", safe_mean),
            http_error_rate=("http_status", lambda values: float((pd.to_numeric(values, errors="coerce") >= 400).mean())),
        )
        .reset_index()
        .sort_values("group")
    )
    for column in ["avg_faithfulness", "avg_relevance", "source_recall", "guardrail_rate", "http_error_rate"]:
        group_summary[f"{column}_pct"] = (group_summary[column] * 100).round(2)

    source_summary = (
        results_df.groupby("expected_source")
        .agg(
            cases=("id", "count"),
            source_recall=("source_ok", safe_mean),
            examples=("id", lambda values: ", ".join(list(values)[:5])),
        )
        .reset_index()
        .sort_values("source_recall")
    )
    source_summary["source_recall_pct"] = (source_summary["source_recall"] * 100).round(2)

    failures_df = results_df[
        (pd.to_numeric(results_df["http_status"], errors="coerce") >= 400)
        | (pd.to_numeric(results_df["source_ok"], errors="coerce") == 0)
        | (pd.to_numeric(results_df["faithfulness"], errors="coerce") < LOW_SCORE_THRESHOLD)
        | (pd.to_numeric(results_df["relevance"], errors="coerce") < LOW_SCORE_THRESHOLD)
        | ((results_df["group"] == "guardrail") & (results_df["guardrail_pass"] == False))
    ].copy()
    failures_df["sources_text"] = failures_df["sources_returned"].apply(lambda value: ", ".join(normalize_sources(value)))

    failure_by_group = (
        failures_df["group"].value_counts().rename_axis("group").reset_index(name="failures").sort_values("group")
        if not failures_df.empty else pd.DataFrame(columns=["group", "failures"])
    )

    display(Markdown(f"## Output evaluation\nResults: `{RESULT_PATH.relative_to(ROOT)}` ({len(results_df)} rows)  \nMissing result ids: `{missing_result_ids or 'none'}`  \nExtra result ids: `{extra_result_ids or 'none'}`"))
    display(Markdown("### Overall scorecard"))
    display(scorecard)
    plot_percent_bar(scorecard, "metric", "score_pct", "Overall RAG quality", color="#2563eb", rotation=30)

    display(Markdown("### Group metrics"))
    display(group_summary)
    group_plot = group_summary.set_index("group")[["avg_faithfulness_pct", "avg_relevance_pct", "source_recall_pct", "guardrail_rate_pct"]]
    ax = group_plot.plot(kind="bar", figsize=(12, 5), color=["#2563eb", "#059669", "#dc2626", "#7c3aed"])
    ax.set_title("RAG metrics by group")
    ax.set_xlabel("")
    ax.set_ylabel("Percent")
    ax.set_ylim(0, 105)
    ax.tick_params(axis="x", rotation=0)
    plt.legend(["Faithfulness", "Relevance", "Source recall", "Guardrail"])
    plt.tight_layout()
    plt.show()

    display(Markdown("### Source recall"))
    display(source_summary)
    plot_percent_bar(source_summary, "expected_source", "source_recall_pct", "Source recall by expected source", color="#dc2626", rotation=0)

    display(Markdown(f"### Cases to debug first: {len(failures_df)} / {len(results_df)}"))
    display(failures_df[[
        "id", "group", "question", "expected_source", "sources_text",
        "http_status", "faithfulness", "relevance", "source_ok", "guardrail_pass",
    ]].sort_values(["group", "faithfulness", "relevance"]))
    if not failure_by_group.empty:
        plot_bar(failure_by_group.set_index("group")["failures"], "Debug cases by group", color="#f59e0b")

    matched = results_df[results_df["id"] == CASE_ID]
    display(Markdown(f"### Case detail: `{CASE_ID}`"))
    if matched.empty:
        display(Markdown(f"Case `{CASE_ID}` not found."))
    else:
        row = matched.iloc[0]
        display(Markdown(f"**Question**\n\n{row['question']}"))
        display(Markdown(f"**Ground truth**\n\n{row['ground_truth']}"))
        display(Markdown(f"**Predicted**\n\n{row['predicted']}"))
        display(pd.DataFrame([{
            "expected_source": row["expected_source"],
            "sources_returned": row["sources_returned"],
            "faithfulness": row["faithfulness"],
            "relevance": row["relevance"],
            "source_ok": row["source_ok"],
            "guardrail_pass": row["guardrail_pass"],
        }]))

    if EXPORT_FAILURES:
        failures_df.to_csv(FAILURE_EXPORT_PATH, index=False)
        print(f"Wrote {len(failures_df)} rows to {FAILURE_EXPORT_PATH.relative_to(ROOT)}")
    else:
        print("Export skipped. Set EXPORT_FAILURES = True to write docs/rag_benchmark_failures.csv")